# Capital budgeting: choosing among investments under a cash limit

Star Oil has five investments available. Each one has a net present value, and each one demands cash
now and again next year. There is 40 million available now and 20 million next year.

You cannot take all five — the cash runs out. **Which do you take?**

The question sounds like ranking, and ranking is the wrong answer. The investment with the highest
NPV may be the one that eats the whole budget; the investment with the best NPV *per dollar* may be
the one that starves the others of next year's cash. Only solving the whole thing at once respects
both constraints simultaneously.

This notebook builds that model by hand, then asks a second question that changes the answer
substantially: **what if you cannot buy half an investment?**

## Licence setup

Same pattern as every notebook here: three secrets named, none contained.

In [1]:
import gurobipy as gp

env = None
try:
    from google.colab import userdata
    try:
        params = {
            "WLSACCESSID": userdata.get("GRB_WLSACCESSID"),
            "WLSSECRET":   userdata.get("GRB_WLSSECRET"),
            "LICENSEID":   int(userdata.get("GRB_LICENSEID")),
        }
    except userdata.SecretNotFoundError:
        raise SystemExit(
            "Add GRB_WLSACCESSID, GRB_WLSSECRET and GRB_LICENSEID as Colab Secrets "
            "(key icon, left sidebar), then re-run this cell."
        )
    env = gp.Env(params=params)
    print("licence: Colab Secrets (WLS)")
except ImportError:
    env = gp.Env()
    print("licence: local gurobi.lic")

Set parameter Username


Set parameter LicenseID to value <removed>


Academic license - for non-commercial use only - expires 2026-12-04


licence: local gurobi.lic


## The investment table

Five investments, each with an NPV and two cash outflows. This is **instance data** — five rows
indexed by investment, with values the narration never names individually — so it lives in a file
that both this notebook and the package read.

That matters here more than usual. The version of this example these notebooks came from typed these
numbers **twice**: once into a dictionary and again, by hand, into the constraint expressions
(`11*x[1,0] + 53*x[2,0] + ...`). Editing the dictionary changed nothing about the model.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
from orteach.capital_budgeting import load_investments

investments = load_investments()

print(f"{'inv':>4} {'NPV':>8} {'cost now':>10} {'cost next':>10}")
for r in investments:
    print(f"{r['investment']:>4} {r['npv']:8.1f} {r['cost_t0']:10.1f} {r['cost_t1']:10.1f}")

 inv      NPV   cost now  cost next
   1     13.0       11.0        3.0
   2     16.0       53.0        6.0
   3     16.0        5.0        5.0
   4     14.0        5.0        1.0
   5     39.0       29.0       34.0


The two budgets are different in kind: they are **knobs**, single numbers the narration explains, so
they are written out here where you can see and change them.

In [3]:
budget_t0 = 40.0   # cash available now
budget_t1 = 20.0   # cash available next year

print(f"total cash across both periods : {budget_t0 + budget_t1:.0f}")
print(f"cost of buying all five        : "
      f"{sum(r['cost_t0'] + r['cost_t1'] for r in investments):.0f}")

total cash across both periods : 60
cost of buying all five        : 152


Buying everything would cost far more than is available, which is why this is a problem at all.

## Before you model: what does ranking suggest?

The instinct is to rank by NPV per dollar and buy down the list. Compute that ranking — it will be
useful later precisely because it turns out to be wrong.

**Predict:** which investment does this ranking put first, and do you think the optimal plan takes it
in full?

In [4]:
print(f"{'inv':>4} {'NPV':>8} {'total cost':>11} {'NPV per $':>11}")
for r in sorted(investments, key=lambda r: -r["npv"] / (r["cost_t0"] + r["cost_t1"])):
    total = r["cost_t0"] + r["cost_t1"]
    print(f"{r['investment']:>4} {r['npv']:8.1f} {total:11.1f} {r['npv']/total:11.3f}")

 inv      NPV  total cost   NPV per $
   4     14.0         6.0       2.333
   3     16.0        10.0       1.600
   1     13.0        14.0       0.929
   5     39.0        63.0       0.619
   2     16.0        59.0       0.271


## The model, one piece at a time

One decision variable per investment: what **fraction** of it to buy. Star Oil can take a partial
stake, so these are continuous.

**The bound is the whole problem.** `ub=1.0` says you cannot buy an investment more than once. Drop
it and the model happily buys three copies of the best-value project and reports an NPV nearly double
the truth — an answer that is arithmetically correct and financially meaningless.

In [5]:
m = gp.Model(env=env)
m.Params.OutputFlag = 0
m.ModelSense = gp.GRB.MAXIMIZE

take = {}
for r in investments:
    take[r["investment"]] = m.addVar(lb=0.0, ub=1.0, obj=r["npv"],
                                     name=f"take[{r['investment']}]")
m.update()

print(f"{m.NumVars} variables, each bounded to [0, 1]")

5 variables, each bounded to [0, 1]


Now the two cash constraints. Each is built by summing over the table — so the numbers come from the
file, and there is exactly one copy of them.

In [6]:
c0 = m.addConstr(gp.quicksum(r["cost_t0"] * take[r["investment"]] for r in investments)
                 <= budget_t0, name="budget_t0")
c1 = m.addConstr(gp.quicksum(r["cost_t1"] * take[r["investment"]] for r in investments)
                 <= budget_t1, name="budget_t1")
m.update()

print(f"constraints: {m.NumConstrs}")
assert m.NumConstrs == 2, "expected exactly two budget constraints"
print("now:  ", m.getRow(c0))
print("next: ", m.getRow(c1))

constraints: 2
now:   11.0 take[1] + 53.0 take[2] + 5.0 take[3] + 5.0 take[4] + 29.0 take[5]
next:  3.0 take[1] + 6.0 take[2] + 5.0 take[3] + take[4] + 34.0 take[5]


**Predict before solving.** Your ranking above put one investment first. Will the optimal plan take
it in full? Will it take *any* investment in full? And will it spend all of both budgets?

In [7]:
m.optimize()

lp_npv = m.ObjVal
lp_take = {k: v.X for k, v in take.items()}

print(f"optimal NPV : {lp_npv:.3f}\n")
print(f"{'inv':>4} {'fraction':>10}")
for k, v in lp_take.items():
    print(f"{k:>4} {v:10.3f}")

optimal NPV : 57.449

 inv   fraction
   1      1.000
   2      0.201
   3      1.000
   4      1.000
   5      0.288


Three investments are taken whole and two are taken partially. Check what that did to the cash.

In [8]:
spend_0 = sum(r["cost_t0"] * lp_take[r["investment"]] for r in investments)
spend_1 = sum(r["cost_t1"] * lp_take[r["investment"]] for r in investments)

print(f"spent now  : {spend_0:6.2f} of {budget_t0:.0f}")
print(f"spent next : {spend_1:6.2f} of {budget_t1:.0f}")

spent now  :  40.00 of 40
spent next :  20.00 of 20


Both budgets are spent to the last dollar. That is why ranking fails: the plan is not "take the best
until the money runs out", it is a balance struck between two constraints at once, and the partial
stakes are how the model uses the last of each budget.

## The second question: what if you cannot buy half a project?

Partial stakes are sometimes real and sometimes a modelling fiction. If each investment is a drilling
programme you either fund or do not, the fractions are meaningless and the answer above is not
executable.

One change: the variables become binary. Everything else is identical.

**Predict — this is the one worth writing down.** The fractional plan earned 57.4. Will the
all-or-nothing plan earn a little less, or a lot less? And will it still spend both budgets?

In [9]:
m2 = gp.Model(env=env)
m2.Params.OutputFlag = 0
m2.ModelSense = gp.GRB.MAXIMIZE

take2 = {}
for r in investments:
    take2[r["investment"]] = m2.addVar(vtype=gp.GRB.BINARY, obj=r["npv"],
                                       name=f"take[{r['investment']}]")
m2.addConstr(gp.quicksum(r["cost_t0"] * take2[r["investment"]] for r in investments)
             <= budget_t0, name="budget_t0")
m2.addConstr(gp.quicksum(r["cost_t1"] * take2[r["investment"]] for r in investments)
             <= budget_t1, name="budget_t1")
m2.optimize()

ip_npv = m2.ObjVal
ip_take = {k: v.X for k, v in take2.items()}

print(f"optimal NPV : {ip_npv:.3f}\n")
print(f"{'inv':>4} {'take':>6}")
for k, v in ip_take.items():
    print(f"{k:>4} {'yes' if v > 0.5 else 'no':>6}")

optimal NPV : 43.000

 inv   take
   1    yes
   2     no
   3    yes
   4    yes
   5     no


Now look at the cash, which is the surprising part.

In [10]:
ip_spend_0 = sum(r["cost_t0"] * ip_take[r["investment"]] for r in investments)
ip_spend_1 = sum(r["cost_t1"] * ip_take[r["investment"]] for r in investments)

print(f"{'':12} {'now':>14} {'next':>14} {'NPV':>10}")
print(f"{'fractional':12} {spend_0:8.2f} / {budget_t0:<3.0f} {spend_1:8.2f} / {budget_t1:<3.0f} {lp_npv:10.3f}")
print(f"{'all or none':12} {ip_spend_0:8.2f} / {budget_t0:<3.0f} {ip_spend_1:8.2f} / {budget_t1:<3.0f} {ip_npv:10.3f}")
print()
print(f"cash left unspent now  : {budget_t0 - ip_spend_0:6.2f}")
print(f"cash left unspent next : {budget_t1 - ip_spend_1:6.2f}")

                        now           next        NPV
fractional      40.00 / 40     20.00 / 20      57.449
all or none     21.00 / 40      9.00 / 20      43.000

cash left unspent now  :  19.00
cash left unspent next :  11.00


The all-or-nothing plan **leaves nearly half the money on the table** — not through carelessness, but
because no combination of whole investments fits the budgets any better. There is no fifth project to
buy with the remainder.

The gap between the two answers is the **cost of indivisibility**: what it costs that investments come
in whole units.

In [11]:
gap = lp_npv - ip_npv
print(f"fractional NPV  : {lp_npv:8.3f}")
print(f"all-or-none NPV : {ip_npv:8.3f}")
print(f"cost of indivisibility : {gap:.3f}   ({100*gap/lp_npv:.1f}% of the LP value)")
assert ip_npv <= lp_npv + 1e-6, "the integer answer cannot beat its own relaxation"
print("\nrelaxation bounds the integer answer, as it must")

fractional NPV  :   57.449
all-or-none NPV :   43.000
cost of indivisibility : 14.449   (25.2% of the LP value)

relaxation bounds the integer answer, as it must


That inequality is worth stating as a rule rather than an observation. The integer problem is the LP
**plus** extra restrictions, so its answer can never be better. The LP value is therefore always an
optimistic bound — useful for knowing how much you might still be leaving behind, and never
achievable unless it happens to come out whole on its own.

---

# Now the streamlined version

Both models differ by one argument — whether the variables are continuous or binary — so they belong
in one function with a flag, now that you have written each out separately.

`orteach.capital_budgeting` takes the investment table **as an argument** and never reads the file
itself, so editing a row above flows into both the hand-built model and the check below.

In [12]:
from orteach import capital_budgeting as cbg

pkg_lp = cbg.solve_fractional(investments, budget_t0, budget_t1, env=env)
pkg_ip = cbg.solve_all_or_none(investments, budget_t0, budget_t1, env=env)

print(f"{pkg_lp.label:20} NPV {pkg_lp.npv:8.3f}")
print(f"{pkg_ip.label:20} NPV {pkg_ip.npv:8.3f}")
print(f"{'indivisibility':20}     {cbg.cost_of_indivisibility(pkg_lp, pkg_ip):8.3f}")

fractional (LP)      NPV   57.449
all or none (MIP)    NPV   43.000
indivisibility             14.449


## The agreement assertion

Same model, built twice on purpose. The check compares every number the notebook produced by hand
against the package's, including each investment fraction — not just the objective, because two
different plans can share an NPV.

In [13]:
checks = [("LP NPV", lp_npv, pkg_lp.npv), ("IP NPV", ip_npv, pkg_ip.npv)]
for r in investments:
    k = r["investment"]
    checks.append((f"LP take[{k}]", lp_take[k], pkg_lp.fractions[k]))
    checks.append((f"IP take[{k}]", ip_take[k], pkg_ip.fractions[k]))

worst = 0.0
for name, hand, packaged in checks:
    rel = abs(packaged - hand) / max(abs(hand), 1.0)
    worst = max(worst, rel)
    print(f"{name:14} hand {hand:10.6f}   package {packaged:10.6f}   rel {rel:.2e}")

assert worst < 1e-9, f"notebook and package disagree by {worst:.2e}"
print(f"\nnotebook and package agree to {worst:.1e}")

LP NPV         hand  57.449017   package  57.449017   rel 0.00e+00
IP NPV         hand  43.000000   package  43.000000   rel 0.00e+00
LP take[1]     hand   1.000000   package   1.000000   rel 0.00e+00
IP take[1]     hand   1.000000   package   1.000000   rel 0.00e+00
LP take[2]     hand   0.200860   package   0.200860   rel 0.00e+00
IP take[2]     hand   0.000000   package   0.000000   rel 0.00e+00
LP take[3]     hand   1.000000   package   1.000000   rel 0.00e+00
IP take[3]     hand   1.000000   package   1.000000   rel 0.00e+00
LP take[4]     hand   1.000000   package   1.000000   rel 0.00e+00
IP take[4]     hand   1.000000   package   1.000000   rel 0.00e+00
LP take[5]     hand   0.288084   package   0.288084   rel 0.00e+00
IP take[5]     hand   0.000000   package   0.000000   rel 0.00e+00

notebook and package agree to 0.0e+00


---

## Where to take this next

- Raise `budget_t1` from 20 to 30 and re-run. Does the all-or-nothing plan change, and does the cost
  of indivisibility grow or shrink?
- The unspent cash in the integer plan is real money. What would you have to add to the model to let
  it earn something — and does that change which projects get funded?
- The LP bound was 57.4 and the achievable answer 43.0. In a larger problem you cannot solve exactly,
  that gap is all you know about how good your answer is. What would make it tighter?